# Retail POS Exploratory Data Analysis (EDA)

This notebook explores cleaned retail POS data for trend, seasonality, product performance, and price-demand behavior.

In [ ]:
# Imports and plot style
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading

In [ ]:
# Locate and load cleaned dataset
candidate_paths = [
    Path('data/processed/cleaned_pos_data.csv'),
    Path('../data/processed/cleaned_pos_data.csv'),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('Could not find cleaned_pos_data.csv in expected locations.')

df = pd.read_csv(data_path)
print(f'Loaded data from: {data_path.resolve()}')
print(f'Shape: {df.shape}')

In [ ]:
# Preview rows
df.head()

In [ ]:
# Data information
df.info()

In [ ]:
# Ensure Date is datetime
if 'Date' not in df.columns:
    raise KeyError("Required column 'Date' is missing from the dataset.")

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date']).copy()

# Validate Quantity column
if 'Quantity' not in df.columns:
    raise KeyError("Required column 'Quantity' is missing from the dataset.")

## 2. Daily Sales Trend
Total Quantity sold per day

In [ ]:
daily_sales = (
    df.groupby(df['Date'].dt.date)['Quantity']
    .sum()
    .reset_index(name='TotalQuantity')
)
daily_sales['Date'] = pd.to_datetime(daily_sales['Date'])

plt.figure(figsize=(14, 6))
sns.lineplot(data=daily_sales, x='Date', y='TotalQuantity', marker='o', label='Total Quantity')
plt.title('Daily Sales Trend: Total Quantity Sold per Day')
plt.xlabel('Date')
plt.ylabel('Total Quantity Sold')
plt.legend(title='Metric')
plt.tight_layout()
plt.show()

## 3. Seasonality Analysis
Average sales by Day of Week (Monday-Sunday)

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
df['DayOfWeek'] = pd.Categorical(df['Date'].dt.day_name(), categories=day_order, ordered=True)
avg_sales_dow = (
    df.groupby('DayOfWeek', observed=False)['Quantity']
    .mean()
    .reset_index(name='AverageQuantity')
)

plt.figure(figsize=(12, 6))
sns.barplot(data=avg_sales_dow, x='DayOfWeek', y='AverageQuantity', hue='DayOfWeek', palette='viridis', dodge=False, legend=False)
plt.title('Seasonality: Average Quantity Sold by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Average Quantity Sold')
plt.legend(['Average Quantity'], title='Metric')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Product Performance
Top 10 products by total quantity sold

In [ ]:
product_col = 'ProductName' if 'ProductName' in df.columns else 'Item_ID'
if product_col not in df.columns:
    raise KeyError("Neither 'ProductName' nor 'Item_ID' is available in the dataset.")

top_products = (
    df.groupby(product_col)['Quantity']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index(name='TotalQuantity')
)

plt.figure(figsize=(12, 7))
sns.barplot(data=top_products, x='TotalQuantity', y=product_col, hue=product_col, palette='mako', dodge=False, legend=False)
plt.title(f'Top 10 Selling Products by Total Quantity ({product_col})')
plt.xlabel('Total Quantity Sold')
plt.ylabel('Product')
plt.legend(['Top Products'], title='Category')
plt.tight_layout()
plt.show()

## 5. Price vs. Demand
Scatter plot and correlation between UnitPrice and Quantity

In [ ]:
price_col = 'UnitPrice' if 'UnitPrice' in df.columns else 'SellingPrice'
if price_col not in df.columns:
    raise KeyError("Neither 'UnitPrice' nor 'SellingPrice' is available in the dataset.")

corr_value = df[[price_col, 'Quantity']].corr().loc[price_col, 'Quantity']

plt.figure(figsize=(12, 6))
sns.scatterplot(data=df, x=price_col, y='Quantity', alpha=0.5, label='Transactions')
plt.title(f'Price vs. Demand: {price_col} vs Quantity (Correlation = {corr_value:.2f})')
plt.xlabel(price_col)
plt.ylabel('Quantity Sold')
plt.legend(title='Data Points')
plt.tight_layout()
plt.show()

print(f'Correlation between {price_col} and Quantity: {corr_value:.4f}')

## 6. Feature Engineering Prep
Create Is_Weekend and Month from Date

In [ ]:
df['Is_Weekend'] = df['Date'].dt.dayofweek >= 5
df['Month'] = df['Date'].dt.month

df[['Date', 'Is_Weekend', 'Month']].head()

In [ ]:
# Final quick check of engineered columns and summary
print('Is_Weekend distribution:')
print(df['Is_Weekend'].value_counts(dropna=False))
print('\nMonth distribution:')
print(df['Month'].value_counts(dropna=False).sort_index())

df.head()